# ============================================================
# M1 ACCOUNT A — RECORD 114 LEAD ORIENTATION CHECK
# ============================================================
#
# PURPOSE:
# Check whether Account A has the same lead reversal
# previously identified in Account B for record 114.
#
# We will:
# 1. Locate the existing processed dataset.
# 2. Locate the original ECG signal and annotation files.
# 3. Reconstruct record 114 from the original files.
# 4. Compare the reconstructed beats with processed data.
# 5. Test normal and swapped lead arrangements.
#
# IMPORTANT:
# This is a diagnostic notebook only.
# It does not modify the dataset or train models.
# ============================================================

import os
import re
import hashlib
import unicodedata

import numpy as np
import pandas as pd

from collections import Counter

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print("Account A — Record 114 diagnostic started.")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

In [1]:
# ============================================================
# M1 ACCOUNT A — RECORD 114 LEAD ORIENTATION CHECK
# ============================================================
#
# PURPOSE:
# Check whether Account A has the same lead reversal
# previously identified in Account B for record 114.
#
# We will:
# 1. Locate the existing processed dataset.
# 2. Locate the original ECG signal and annotation files.
# 3. Reconstruct record 114 from the original files.
# 4. Compare the reconstructed beats with processed data.
# 5. Test normal and swapped lead arrangements.
#
# IMPORTANT:
# This is a diagnostic notebook only.
# It does not modify the dataset or train models.
# ============================================================

import os
import re
import hashlib
import unicodedata

import numpy as np
import pandas as pd

from collections import Counter

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print("Account A — Record 114 diagnostic started.")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

Account A — Record 114 diagnostic started.
NumPy version: 2.1.3
Pandas version: 2.3.3


In [5]:
# ============================================================
# CELL 2 — AUTOMATIC INPUT FILE DISCOVERY
# ============================================================

INPUT_ROOT = "/kaggle/input"

def clean_column_name(name):
    """Remove surrounding whitespace and quotation marks."""
    name = unicodedata.normalize("NFKC", str(name))
    return name.strip().strip('"').strip("'").strip()


def normalise_name(name):
    """Normalise a column name for reliable identification."""
    name = clean_column_name(name).casefold()
    return re.sub(r"[^a-z0-9#]", "", name)


# Expected processed waveform columns
MLII_COLUMNS = [f"MLII_{i}" for i in range(201)]
LEAD2_COLUMNS = [f"Lead2_{i}" for i in range(201)]
WAVEFORM_COLUMNS = MLII_COLUMNS + LEAD2_COLUMNS

assert len(WAVEFORM_COLUMNS) == 402

processed_candidates = []
signal_candidates = []
annotation_candidates = []

for root, dirs, files in os.walk(INPUT_ROOT):

    for filename in files:

        full_path = os.path.join(root, filename)
        lower_name = filename.lower()
        lower_path = full_path.lower()

        # Identify original record 114 signal file.
        if lower_name == "114.csv":
            signal_candidates.append(full_path)

        # Identify original record 114 annotation file.
        if (
            "annotation" in lower_path
            and re.match(r"^114(?:$|[^0-9])", os.path.splitext(lower_name)[0])
        ):
            annotation_candidates.append(full_path)

        # Identify processed dataset by its column structure.
        if lower_name.endswith(".csv"):

            try:
                header_df = pd.read_csv(
                    full_path,
                    nrows=0
                )

                cleaned_headers = [
                    clean_column_name(c)
                    for c in header_df.columns
                ]

                header_set = set(cleaned_headers)

                required = {
                    "Record_ID",
                    "Label",
                    "MLII_0",
                    "MLII_200",
                    "Lead2_0",
                    "Lead2_200"
                }

                if required.issubset(header_set):
                    processed_candidates.append(full_path)

            except Exception:
                pass


print("PROCESSED DATASET CANDIDATES:")
for p in processed_candidates:
    print(p)

print("\nRECORD 114 SIGNAL FILES:")
for p in signal_candidates:
    print(p)

print("\nRECORD 114 ANNOTATION FILES:")
for p in annotation_candidates:
    print(p)


# Select the processed dataset.
if not processed_candidates:
    raise FileNotFoundError(
        "Could not identify the processed ECG dataset."
    )

if len(processed_candidates) == 1:
    processed_path = processed_candidates[0]

else:
    preferred = [
        p for p in processed_candidates
        if "validated_aami_dataset" in os.path.basename(p).lower()
    ]

    if len(preferred) == 1:
        processed_path = preferred[0]
    else:
        # Prefer the largest matching dataset file.
        processed_path = max(
            processed_candidates,
            key=os.path.getsize
        )


# Select the original signal file.
if not signal_candidates:
    raise FileNotFoundError(
        "Could not find original 114.csv."
    )

preferred_signal = [
    p for p in signal_candidates
    if "signals" in p.lower().split(os.sep)
]

if len(preferred_signal) == 1:
    signal_path = preferred_signal[0]
elif len(signal_candidates) == 1:
    signal_path = signal_candidates[0]
else:
    raise RuntimeError(
        "Multiple signal files found. Please inspect the paths above."
    )


# Select the original annotation file.
if not annotation_candidates:
    raise FileNotFoundError(
        "Could not find original 114.txt."
    )

preferred_annotation = [
    p for p in annotation_candidates
    if "annotations" in p.lower().split(os.sep)
]

if len(preferred_annotation) == 1:
    annotation_path = preferred_annotation[0]
elif len(annotation_candidates) == 1:
    annotation_path = annotation_candidates[0]
else:
    raise RuntimeError(
        "Multiple annotation files found. Please inspect the paths above."
    )


print("\n" + "=" * 65)
print("SELECTED FILES")
print("=" * 65)

print("Processed dataset:", processed_path)
print("Original signal:", signal_path)
print("Original annotation:", annotation_path)

PROCESSED DATASET CANDIDATES:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv

RECORD 114 SIGNAL FILES:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Signals/114.csv

RECORD 114 ANNOTATION FILES:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/114annotations.txt

SELECTED FILES
Processed dataset: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv
Original signal: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Signals/114.csv
Original annotation: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/114annotations.txt


In [7]:
# ============================================================
# CELL 3 — LOAD AND INSPECT THE THREE FILES
# ============================================================

# ------------------------------------------------------------
# Load processed dataset
# ------------------------------------------------------------

processed_df = pd.read_csv(processed_path)

processed_df.columns = [
    clean_column_name(c)
    for c in processed_df.columns
]

print("PROCESSED DATASET")
print("Shape:", processed_df.shape)
print("Record count:", processed_df["Record_ID"].nunique())

# Confirm all 402 signal columns exist.
missing_columns = [
    c for c in WAVEFORM_COLUMNS
    if c not in processed_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing waveform columns: {missing_columns}"
    )

processed_df["Record_ID"] = pd.to_numeric(
    processed_df["Record_ID"],
    errors="raise"
).astype(int)

processed_df["Label"] = (
    processed_df["Label"].astype(str).str.strip()
)


# ------------------------------------------------------------
# Load original record 114 signal
# ------------------------------------------------------------

signal_df = pd.read_csv(signal_path)

signal_df.columns = [
    clean_column_name(c)
    for c in signal_df.columns
]

def find_column_by_normalised_name(df, wanted):
    matches = [
        c for c in df.columns
        if normalise_name(c) == wanted
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one column matching {wanted}, "
            f"found {matches}"
        )

    return matches[0]


sample_col = find_column_by_normalised_name(
    signal_df, "sample#"
)

mlii_col = find_column_by_normalised_name(
    signal_df, "mlii"
)

other_leads = [
    c for c in signal_df.columns
    if c not in [sample_col, mlii_col]
]

if len(other_leads) != 1:
    raise ValueError(
        f"Expected exactly one second lead, found {other_leads}"
    )

second_lead_col = other_leads[0]

signal_df[sample_col] = pd.to_numeric(
    signal_df[sample_col],
    errors="raise"
).astype(int)

signal_df[mlii_col] = pd.to_numeric(
    signal_df[mlii_col],
    errors="raise"
)

signal_df[second_lead_col] = pd.to_numeric(
    signal_df[second_lead_col],
    errors="raise"
)


print("\nORIGINAL SIGNAL FILE")
print("Shape:", signal_df.shape)
print("Sample index column:", sample_col)
print("MLII column:", mlii_col)
print("Second lead:", second_lead_col)

display(
    signal_df[
        [sample_col, mlii_col, second_lead_col]
    ].head(10)
)


# ------------------------------------------------------------
# Load original annotation file
# ------------------------------------------------------------

print("\nANNOTATION FILE — FIRST 10 LINES")

with open(
    annotation_path,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for i, line in enumerate(f):
        if i >= 10:
            break

        print(repr(line.rstrip("\n")))


print("\nFiles loaded successfully.")

PROCESSED DATASET
Shape: (106870, 404)
Record count: 46

ORIGINAL SIGNAL FILE
Shape: (650000, 3)
Sample index column: sample #
MLII column: MLII
Second lead: V5


,sample #,MLII,V5
0,0,1027,1015
1,1,1027,1015
2,2,1027,1015
3,3,1027,1015
4,4,1027,1015
5,5,1027,1015
6,6,1027,1015
7,7,1027,1015
8,8,1027,1015
9,9,1032,1015



ANNOTATION FILE — FIRST 10 LINES
'      Time   Sample #  Type  Sub Chan  Num\tAux'
'    0:00.058       21     +    0    0    0\t(N'
'    0:00.861      310     N    0    0    0'
'    0:02.017      726     N    0    0    0'
'    0:03.175     1143     N    0    0    0'
'    0:04.314     1553     N    0    0    0'
'    0:05.458     1965     N    0    0    0'
'    0:06.550     2358     N    0    0    0'
'    0:07.617     2742     N    0    0    0'
'    0:08.794     3166     N    0    0    0'

Files loaded successfully.


In [8]:

# ================================================================
# CELL 4 — CLEAN ORIGINAL SIGNAL HEADER INSPECTION
# ================================================================

signal_preview = pd.read_csv(
    signal_path,
    nrows=5
)

# Remove unnecessary quote characters from column names.
signal_preview.columns = [
    str(column).strip().strip("'").strip('"').strip()
    for column in signal_preview.columns
]

print("CLEANED SIGNAL COLUMN NAMES:")
print(signal_preview.columns.tolist())

print("\nFIRST 5 ORIGINAL SIGNAL ROWS:")
display(signal_preview)

print("\nSIGNAL SAMPLE COLUMN CHECK:")

if "sample #" in signal_preview.columns:
    print("PASS: Original sample position column found.")
else:
    print("WARNING: Sample position column not found.")

print("\nCELL 4 COMPLETE.")


CLEANED SIGNAL COLUMN NAMES:
['sample #', 'V5', 'MLII']

FIRST 5 ORIGINAL SIGNAL ROWS:


,sample #,V5,MLII
0,0,1015,1027
1,1,1015,1027
2,2,1015,1027
3,3,1015,1027
4,4,1015,1027



SIGNAL SAMPLE COLUMN CHECK:
PASS: Original sample position column found.

CELL 4 COMPLETE.


In [ ]:

# ================================================================
# M1-B CELL 6 — LOAD THE EXISTING PROCESSED DATASET
# ================================================================

import pandas as pd
import numpy as np
import re
import os
import json
import hashlib
from pathlib import Path
from collections import Counter

print("=" * 70)
print("M1-B: LOAD EXISTING PROCESSED DATASET")
print("=" * 70)

# Use the processed dataset located in M1-A.
# No new dataset upload is required.

processed_df = pd.read_csv(
    processed_path,
    low_memory=False
)

# Clean possible quotation marks and spaces in headers.
processed_df.columns = [
    str(c).strip().strip("'").strip('"').strip()
    for c in processed_df.columns
]

# Check for duplicate column names.
if processed_df.columns.duplicated().any():
    raise ValueError("Duplicate columns found after header cleaning.")

# Normalize Record_ID.
processed_df["Record_ID"] = pd.to_numeric(
    processed_df["Record_ID"],
    errors="raise"
).astype(int)

processed_df["Label"] = (
    processed_df["Label"].astype(str).str.strip()
)

# Expected waveform column names.
MLII_COLUMNS = [f"MLII_{i}" for i in range(201)]
LEAD2_COLUMNS = [f"Lead2_{i}" for i in range(201)]

WAVEFORM_COLUMNS = MLII_COLUMNS + LEAD2_COLUMNS

missing_columns = [
    c for c in ["Record_ID", "Label"] + WAVEFORM_COLUMNS
    if c not in processed_df.columns
]

if missing_columns:
    raise ValueError(
        "Required columns missing from the processed dataset:\n"
        + str(missing_columns[:20])
    )

# Convert waveform values to numeric.
processed_df[WAVEFORM_COLUMNS] = (
    processed_df[WAVEFORM_COLUMNS]
    .apply(pd.to_numeric, errors="raise")
)

print("Processed dataset loaded.")
print("Shape:", processed_df.shape)
print("Number of recordings:", processed_df["Record_ID"].nunique())
print("Number of rows before AAMI filtering:", len(processed_df))
print("Number of waveform columns:", len(WAVEFORM_COLUMNS))

print("\nRecord IDs:")
print(sorted(processed_df["Record_ID"].unique().tolist()))

print("\nOriginal processed label counts:")
print(processed_df["Label"].value_counts().sort_index())

print("\nCELL 6 COMPLETE.")


In [ ]:

# ================================================================
# M1-B CELL 7 — SOURCE RECORDING ELIGIBILITY AUDIT
# ================================================================

print("=" * 70)
print("M1-B: SOURCE RECORDING ELIGIBILITY AUDIT")
print("=" * 70)

annotation_files = {}
signal_files = {}

# Locate original annotation files.
for path in INPUT_ROOT.rglob("*annotations.txt"):

    match = re.fullmatch(
        r"(\d+)annotations\.txt",
        path.name,
        flags=re.IGNORECASE
    )

    if match:
        record_id = int(match.group(1))

        if record_id in annotation_files:
            raise ValueError(
                f"Duplicate annotation files for record {record_id}."
            )

        annotation_files[record_id] = path

# Locate original signal files.
for path in INPUT_ROOT.rglob("*.csv"):

    match = re.fullmatch(
        r"(\d+)\.csv",
        path.name,
        flags=re.IGNORECASE
    )

    if match:
        record_id = int(match.group(1))

        if record_id in signal_files:
            raise ValueError(
                f"Duplicate signal files for record {record_id}."
            )

        signal_files[record_id] = path

PROCESSED_RECORD_IDS = sorted(
    processed_df["Record_ID"].unique().tolist()
)

processed_id_set = set(PROCESSED_RECORD_IDS)

source_ids = sorted(
    set(annotation_files) | set(signal_files)
)

audit_rows = []

for record_id in source_ids:

    has_annotation = record_id in annotation_files
    has_signal = record_id in signal_files
    has_mlii = False
    signal_leads = []

    if has_signal:

        header = pd.read_csv(
            signal_files[record_id],
            nrows=0
        )

        signal_leads = [
            str(c).strip().strip("'").strip('"').strip()
            for c in header.columns
        ]

        normalized = [c.upper() for c in signal_leads]

        has_mlii = "MLII" in normalized

    if not has_annotation:
        status = "NO_ANNOTATION_FILE"

    elif not has_signal:
        status = "NO_SIGNAL_FILE"

    elif not has_mlii:
        status = "NO_MLII_EXCLUDED"

    else:
        status = "ELIGIBLE"

    audit_rows.append({
        "Record_ID": record_id,
        "Has_Annotation": has_annotation,
        "Has_Signal": has_signal,
        "Has_MLII": has_mlii,
        "Included_In_Processed_Dataset":
            record_id in processed_id_set,
        "Source_Status": status,
        "Signal_Leads": ", ".join(signal_leads)
    })

source_audit_df = pd.DataFrame(audit_rows)

display(source_audit_df)

eligible_ids = set(
    source_audit_df.loc[
        source_audit_df["Source_Status"] == "ELIGIBLE",
        "Record_ID"
    ]
)

excluded_no_mlii = sorted(
    source_audit_df.loc[
        source_audit_df["Source_Status"] == "NO_MLII_EXCLUDED",
        "Record_ID"
    ].tolist()
)

extra_eligible = sorted(
    eligible_ids - processed_id_set
)

missing_eligible = sorted(
    processed_id_set - eligible_ids
)

print("\nSOURCE AUDIT SUMMARY")
print("Annotation files:", len(annotation_files))
print("Signal files:", len(signal_files))
print("Processed recordings:", len(PROCESSED_RECORD_IDS))
print("Eligible source recordings:", len(eligible_ids))
print("Recordings without MLII:", excluded_no_mlii)

print("\nEligible but absent from processed dataset:", extra_eligible)
print("Processed records not source-eligible:", missing_eligible)

# Check that the original eligibility matches the
# recordings actually retained in our processed dataset.

if extra_eligible or missing_eligible:
    raise ValueError(
        "STOP: Source eligibility does not match the "
        "processed dataset. Investigate before extraction."
    )

if len(PROCESSED_RECORD_IDS) != 46:
    raise ValueError(
        f"Expected 46 retained recordings, found "
        f"{len(PROCESSED_RECORD_IDS)}."
    )

if not {201, 202}.issubset(processed_id_set):
    raise ValueError(
        "Records 201 and 202 are not both present."
    )

print("\nRECORDING ELIGIBILITY AUDIT: PASS")
print("All retained recordings have original annotations and MLII.")
print("CELL 7 COMPLETE.")


In [ ]:

# ================================================================
# M1-B CELL 8 — PARSE ANNOTATIONS AND MAP AAMI LABELS
# ================================================================

print("=" * 70)
print("M1-B: ANNOTATION PARSER AND AAMI MAPPING")
print("=" * 70)

ANNOTATION_COLUMNS = [
    "Time", "Sample_Index", "Original_Symbol",
    "Sub", "Chan", "Num", "Aux"
]

AAMI_MAPPING = {
    "N": "N",
    "L": "N",
    "R": "N",
    "e": "N",
    "j": "N",

    "A": "S",
    "a": "S",
    "J": "S",
    "S": "S",

    "V": "V",
    "E": "V",

    "F": "F",

    "/": "Q",
    "f": "Q",
    "Q": "Q"
}
EXPECTED_BEATS_PER_REPEAT = 105047

EXPECTED_CLASS_COUNTS = {
    "N": 90341,
    "S": 2781,
    "V": 7229,
    "F": 802,
    "Q": 3894
}

AAMI_CLASSES = ["N", "S", "V", "F", "Q"]


def parse_annotation_file(path, record_id):

    records = []
    bad_lines = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
        errors="replace"
    ) as f:

        lines = f.readlines()

    # First line is the annotation header.
    for line_number, line in enumerate(lines[1:], start=2):

        if not line.strip():
            continue

        # The annotation TXT uses whitespace-aligned columns.
        # maxsplit=6 keeps the remaining Aux text together.
        parts = re.split(
            r"\s+",
            line.strip(),
            maxsplit=6
        )

        if len(parts) < 6:
            bad_lines.append((line_number, line.rstrip()))
            continue

        parts += [None] * (7 - len(parts))

        try:
            sample_index = int(parts[1])
        except (TypeError, ValueError):
            bad_lines.append((line_number, line.rstrip()))
            continue

        records.append({
            "Record_ID": int(record_id),
            "Time": parts[0],
            "Sample_Index": sample_index,
            "Original_Symbol": parts[2],
            "Sub": parts[3],
            "Chan": parts[4],
            "Num": parts[5],
            "Aux": parts[6]
        })

    annotation = pd.DataFrame(records)

    if annotation.empty:
        raise ValueError(
            f"No annotation rows parsed for record {record_id}."
        )

    annotation["AAMI_Label"] = (
        annotation["Original_Symbol"].map(AAMI_MAPPING)
    )

    return annotation, bad_lines


# Test the parser again against record 100.
test_annotation, test_bad_lines = parse_annotation_file(
    annotation_files[100],
    100
)

print("Record 100 annotation rows:", len(test_annotation))
print("Record 100 parser errors:", len(test_bad_lines))

print("\nOriginal symbol counts:")
print(test_annotation["Original_Symbol"].value_counts())

print("\nAAMI-mapped symbol counts:")
print(
    test_annotation["AAMI_Label"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nFirst 10 mapped annotation rows:")
display(test_annotation.head(10))

print("\nCELL 8 COMPLETE.")


In [ ]:

# ================================================================
# M1-B CELL 9 — REBUILD THE BEAT TABLE FROM ORIGINAL FILES
# ================================================================

print("=" * 70)
print("M1-B: REBUILD ALL 46 RECORDINGS")
print("=" * 70)

WINDOW_BEFORE = 100
WINDOW_AFTER = 100
WINDOW_LENGTH = WINDOW_BEFORE + WINDOW_AFTER + 1

if WINDOW_LENGTH != 201:
    raise ValueError("Unexpected waveform window length.")

# Person ID rule:
# Record 201 and 202 belong to the same person.
def get_person_id(record_id):

    if int(record_id) in [201, 202]:
        return "person_201_202"

    return f"person_{int(record_id)}"


metadata_rows = []
waveform_rows = []

record_audit = []
annotation_audit = []
parser_errors = []

for record_number, record_id in enumerate(
    PROCESSED_RECORD_IDS,
    start=1
):

    print(
        f"\nProcessing record {record_id} "
        f"({record_number}/{len(PROCESSED_RECORD_IDS)})"
    )

    # ------------------------------------------------------------
    # A. Parse original annotations.
    # ------------------------------------------------------------

    annotation, bad_lines = parse_annotation_file(
        annotation_files[record_id],
        record_id
    )

    for line_number, raw_line in bad_lines:
        parser_errors.append({
            "Record_ID": record_id,
            "Line_Number": line_number,
            "Raw_Line": raw_line
        })

    annotation_audit.append({
        "Record_ID": record_id,
        "Annotation_Rows": len(annotation),
        "Parser_Errors": len(bad_lines),
        "Mapped_Beat_Annotations": int(
            annotation["AAMI_Label"].notna().sum()
        ),
        "Unmapped_Annotations": int(
            annotation["AAMI_Label"].isna().sum()
        )
    })

    # Keep only symbols belonging to the five AAMI classes.
    beats = annotation[
        annotation["AAMI_Label"].notna()
    ].copy()

    # Duplicate beat positions require investigation.
    duplicated_positions = beats.duplicated(
        subset=["Sample_Index"],
        keep=False
    )

    if duplicated_positions.any():
        display(
            beats.loc[
                duplicated_positions,
                ["Sample_Index", "Original_Symbol", "AAMI_Label"]
            ].head(10)
        )

        raise ValueError(
            f"Duplicate mapped beat positions in record {record_id}."
        )

    # ------------------------------------------------------------
    # B. Load original signal CSV.
    # ------------------------------------------------------------

    signal = pd.read_csv(
        signal_files[record_id],
        low_memory=False
    )

    # Clean source headers, including quoted column names.
    signal.columns = [
        str(c).strip().strip("'").strip('"').strip()
        for c in signal.columns
    ]

    if signal.columns.duplicated().any():
        raise ValueError(
            f"Duplicate signal columns in record {record_id}."
        )

    sample_candidates = [
        c for c in signal.columns
        if c.lower().strip() == "sample #"
    ]

    if len(sample_candidates) != 1:
        raise ValueError(
            f"Could not uniquely identify sample # in record {record_id}."
        )

    sample_column = sample_candidates[0]

    lead_columns = [
        c for c in signal.columns
        if c != sample_column
    ]

    mlii_candidates = [
        c for c in lead_columns
        if c.upper().strip() == "MLII"
    ]

    if len(mlii_candidates) != 1:
        raise ValueError(
            f"MLII lead not uniquely identified in record {record_id}."
        )

    mlii_column = mlii_candidates[0]

    other_leads = [
        c for c in lead_columns
        if c != mlii_column
    ]

    if len(other_leads) != 1:
        raise ValueError(
            f"Expected exactly one second lead in record {record_id}. "
            f"Found: {other_leads}"
        )

    lead2_column = other_leads[0]

    # Convert sample indices and signal values.
    signal[sample_column] = pd.to_numeric(
        signal[sample_column],
        errors="raise"
    ).astype(np.int64)

    signal[[mlii_column, lead2_column]] = (
        signal[[mlii_column, lead2_column]]
        .apply(pd.to_numeric, errors="raise")
    )

    sample_values = signal[sample_column].to_numpy()

    # Sample positions must be unique and contiguous.
    if len(np.unique(sample_values)) != len(sample_values):
        raise ValueError(
            f"Duplicate sample positions in record {record_id}."
        )

    if not np.all(np.diff(sample_values) == 1):
        raise ValueError(
            f"Signal sample positions are not contiguous in record {record_id}."
        )

    signal_start = int(sample_values[0])
    signal_end = int(sample_values[-1])

    if signal_start != 0:
        raise ValueError(
            f"Expected original signal to start at sample 0 "
            f"for record {record_id}; found {signal_start}."
        )

    signal_values = signal[
        [mlii_column, lead2_column]
    ].to_numpy(dtype=np.float64)

    # ------------------------------------------------------------
    # C. Extract windows around exact annotation positions.
    # ------------------------------------------------------------

    accepted = 0
    boundary_skipped = 0

    for row in beats.itertuples(index=False):

        center = int(row.Sample_Index)

        start = center - WINDOW_BEFORE
        end = center + WINDOW_AFTER

        # Do not pad or invent samples at the signal boundaries.
        if start < signal_start or end > signal_end:
            boundary_skipped += 1
            continue

        start_position = start - signal_start
        end_position = end - signal_start + 1

        window = signal_values[
            start_position:end_position
        ]

        if window.shape != (201, 2):
            raise ValueError(
                f"Incorrect window shape in record {record_id}, "
                f"sample {center}: {window.shape}"
            )

        if not np.isfinite(window).all():
            raise ValueError(
                f"Non-finite signal values in record {record_id}, "
                f"sample {center}."
            )

        mlii_window = window[:, 0]
        lead2_window = window[:, 1]

        metadata_rows.append({
            "Record_ID": int(record_id),
            "Person_ID": get_person_id(record_id),
            "Sample_Index": center,
            "Original_Symbol": row.Original_Symbol,
            "AAMI_Label": row.AAMI_Label
        })

        waveform_rows.append(
            np.concatenate([
                mlii_window,
                lead2_window
            ])
        )

        accepted += 1

    record_audit.append({
        "Record_ID": int(record_id),
        "Person_ID": get_person_id(record_id),
        "Annotation_Beat_Candidates": len(beats),
        "Accepted_Full_Windows": accepted,
        "Boundary_Windows_Skipped": boundary_skipped,
        "MLII_Column": mlii_column,
        "Second_Lead_Column": lead2_column,
        "Signal_Samples": len(signal)
    })

# ------------------------------------------------------------
# D. Assemble the rebuilt beat table.
# ------------------------------------------------------------

metadata_df = pd.DataFrame(metadata_rows)

waveform_matrix = np.vstack(waveform_rows)

waveform_columns = WAVEFORM_COLUMNS

waveform_df = pd.DataFrame(
    waveform_matrix,
    columns=waveform_columns
)

rebuilt_df = pd.concat(
    [
        metadata_df.reset_index(drop=True),
        waveform_df.reset_index(drop=True)
    ],
    axis=1
)

record_audit_df = pd.DataFrame(record_audit)
annotation_audit_df = pd.DataFrame(annotation_audit)
parser_errors_df = pd.DataFrame(parser_errors)

# Sort by record and original sample position.
# This is the natural organization of the reconstructed table,
# NOT the method used to verify waveform equality.
rebuilt_df = rebuilt_df.sort_values(
    ["Record_ID", "Sample_Index"],
    kind="mergesort"
).reset_index(drop=True)
# Verify recording-to-person mapping.

assert rebuilt_df["Record_ID"].nunique() == 46

assert rebuilt_df["Person_ID"].nunique() == 45

assert get_person_id(201) == get_person_id(202)

assert (
    rebuilt_df.loc[
        rebuilt_df["Record_ID"].isin([201, 202]),
        "Person_ID"
    ].nunique() == 1
)

print("PASS: 46 recordings represented.")
print("PASS: 45 distinct people represented.")
print("PASS: Records 201 and 202 share one Person_ID.")
print("\nREBUILD COMPLETED.")
print("Rebuilt beat table shape:", rebuilt_df.shape)
print("Rebuilt beats:", len(rebuilt_df))
print("Unique records:", rebuilt_df["Record_ID"].nunique())
print("Unique people:", rebuilt_df["Person_ID"].nunique())

print("\nAAMI class counts:")
print(
    rebuilt_df["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
)

print("\nRecord-level extraction audit:")
display(record_audit_df.head(10))

print("\nCELL 9 COMPLETE.")


In [ ]:
# ============================================================
# M1-B CELL 10: CORRECTED VALIDATION
# Map original annotation symbols to AAMI classes BEFORE
# comparing the existing processed dataset with reconstruction.
# ============================================================

import hashlib
import numpy as np
import pandas as pd
from collections import Counter

# ------------------------------------------------------------
# 1. Define the official AAMI mapping
# ------------------------------------------------------------

AAMI_MAP = {
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V",
    "F": "F",
    "/": "Q", "f": "Q", "Q": "Q"
}

EXPECTED_COUNTS = {
    "N": 90341,
    "S": 2781,
    "V": 7229,
    "F": 802,
    "Q": 3894
}

AAMI_CLASSES = ["N", "S", "V", "F", "Q"]

# The expected 402 waveform columns, in their original order.
waveform_cols = (
    [f"MLII_{i}" for i in range(201)]
    + [f"Lead2_{i}" for i in range(201)]
)

missing_processed = [
    c for c in waveform_cols if c not in processed_df.columns
]
missing_rebuilt = [
    c for c in waveform_cols if c not in rebuilt_df.columns
]

assert len(waveform_cols) == 402
assert not missing_processed, f"Missing processed columns: {missing_processed}"
assert not missing_rebuilt, f"Missing rebuilt columns: {missing_rebuilt}"

# ------------------------------------------------------------
# 2. Map the original processed labels to AAMI classes
# ------------------------------------------------------------

processed_check = processed_df.copy()

processed_check["Record_ID"] = pd.to_numeric(
    processed_check["Record_ID"], errors="raise"
).astype(int)

processed_check["AAMI_Label"] = (
    processed_check["Label"]
    .astype(str)
    .str.strip()
    .map(AAMI_MAP)
)

# Keep only rows belonging to the five AAMI beat classes.
processed_valid = processed_check[
    processed_check["AAMI_Label"].notna()
].copy()

# ------------------------------------------------------------
# 3. Prepare the reconstructed beat table
# ------------------------------------------------------------

rebuilt_check = rebuilt_df.copy()

rebuilt_check["Record_ID"] = pd.to_numeric(
    rebuilt_check["Record_ID"], errors="raise"
).astype(int)

rebuilt_check["AAMI_Label"] = (
    rebuilt_check["AAMI_Label"]
    .astype(str)
    .str.strip()
)

assert rebuilt_check["AAMI_Label"].isin(AAMI_CLASSES).all()

# ------------------------------------------------------------
# 4. Compare total and class counts
# ------------------------------------------------------------

processed_class_counts = (
    processed_valid["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
)

rebuilt_class_counts = (
    rebuilt_check["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
)

class_count_comparison = pd.DataFrame({
    "Expected": pd.Series(EXPECTED_COUNTS),
    "Existing_Processed": processed_class_counts,
    "Rebuilt_From_Original": rebuilt_class_counts
})

class_count_comparison["Rebuilt_Matches_Existing"] = (
    class_count_comparison["Existing_Processed"]
    == class_count_comparison["Rebuilt_From_Original"]
)

class_count_comparison["Matches_Expected"] = (
    class_count_comparison["Expected"]
    == class_count_comparison["Rebuilt_From_Original"]
)

display(class_count_comparison)

# ------------------------------------------------------------
# 5. Compare record IDs
# ------------------------------------------------------------

processed_record_ids = set(processed_valid["Record_ID"].unique())
rebuilt_record_ids = set(rebuilt_check["Record_ID"].unique())

record_id_sets_pass = (
    processed_record_ids == rebuilt_record_ids
)

print("Processed record count:", len(processed_record_ids))
print("Rebuilt record count:", len(rebuilt_record_ids))
print("Record ID sets match:", record_id_sets_pass)

# ------------------------------------------------------------
# 6. Compare class counts within every record
# ------------------------------------------------------------

processed_record_classes = (
    processed_valid
    .groupby(["Record_ID", "AAMI_Label"])
    .size()
    .rename("Processed_Count")
)

rebuilt_record_classes = (
    rebuilt_check
    .groupby(["Record_ID", "AAMI_Label"])
    .size()
    .rename("Rebuilt_Count")
)

record_level_class_comparison = pd.concat(
    [processed_record_classes, rebuilt_record_classes],
    axis=1
).fillna(0).astype(int)

record_level_class_comparison["Match"] = (
    record_level_class_comparison["Processed_Count"]
    == record_level_class_comparison["Rebuilt_Count"]
)

record_level_class_counts_pass = (
    record_level_class_comparison["Match"].all()
)

print(
    "Record-level class counts match:",
    record_level_class_counts_pass
)

# ------------------------------------------------------------
# 7. Create exact waveform fingerprints
# ------------------------------------------------------------

def make_waveform_counter(df, include_label=False):

    counter = Counter()

    columns = ["Record_ID", "AAMI_Label"] + waveform_cols

    for row in df[columns].itertuples(index=False, name=None):

        record_id = int(row[0])
        aami_label = str(row[1])

        waveform = np.asarray(row[2:], dtype=np.float64)

        if waveform.size != 402:
            raise ValueError(
                f"Expected 402 values, found {waveform.size}"
            )

        digest = hashlib.blake2b(
            waveform.tobytes(),
            digest_size=16
        ).hexdigest()

        if include_label:
            key = (record_id, aami_label, digest)
        else:
            key = (record_id, digest)

        counter[key] += 1

    return counter


print("Creating processed waveform fingerprints...")
processed_waveform_counter = make_waveform_counter(
    processed_valid,
    include_label=False
)

print("Creating rebuilt waveform fingerprints...")
rebuilt_waveform_counter = make_waveform_counter(
    rebuilt_check,
    include_label=False
)

print("Creating processed waveform + label fingerprints...")
processed_label_counter = make_waveform_counter(
    processed_valid,
    include_label=True
)

print("Creating rebuilt waveform + label fingerprints...")
rebuilt_label_counter = make_waveform_counter(
    rebuilt_check,
    include_label=True
)

# ------------------------------------------------------------
# 8. Compare waveform fingerprints
# ------------------------------------------------------------

exact_waveforms_pass = (
    processed_waveform_counter == rebuilt_waveform_counter
)

exact_waveform_labels_pass = (
    processed_label_counter == rebuilt_label_counter
)

# Per-record mismatch details
mismatch_rows = []

for record_id in sorted(
    processed_record_ids | rebuilt_record_ids
):

    p_wave = Counter({
        k: v for k, v in processed_waveform_counter.items()
        if k[0] == record_id
    })

    r_wave = Counter({
        k: v for k, v in rebuilt_waveform_counter.items()
        if k[0] == record_id
    })

    p_label = Counter({
        k: v for k, v in processed_label_counter.items()
        if k[0] == record_id
    })

    r_label = Counter({
        k: v for k, v in rebuilt_label_counter.items()
        if k[0] == record_id
    })

    mismatch_rows.append({
        "Record_ID": record_id,
        "Missing_Waveforms": sum((p_wave - r_wave).values()),
        "Extra_Waveforms": sum((r_wave - p_wave).values()),
        "Missing_Labelled_Rows": sum((p_label - r_label).values()),
        "Extra_Labelled_Rows": sum((r_label - p_label).values())
    })

mismatch_diagnostics = pd.DataFrame(mismatch_rows)

display(mismatch_diagnostics)

# ------------------------------------------------------------
# 9. Final validation checks
# ------------------------------------------------------------

expected_total_pass = (
    len(processed_valid) == 105047
    and len(rebuilt_check) == 105047
)

expected_class_counts_pass = (
    processed_class_counts.to_dict() == EXPECTED_COUNTS
    and rebuilt_class_counts.to_dict() == EXPECTED_COUNTS
)

all_checks = {
    "Expected_total_beat_count": expected_total_pass,
    "Expected_class_counts": expected_class_counts_pass,
    "Record_ID_sets": record_id_sets_pass,
    "Record_level_class_counts": record_level_class_counts_pass,
    "Exact_402_value_waveforms": exact_waveforms_pass,
    "Exact_waveform_and_AAMI_labels": exact_waveform_labels_pass,
    "No_annotation_parser_errors": (
        len(parser_errors_df) == 0
    ),
    "No_duplicate_record_sample_keys": (
        not rebuilt_check.duplicated(
            ["Record_ID", "Sample_Index"]
        ).any()
    )
}

print("\n" + "=" * 65)
print("M1-B CORRECTED VALIDATION RESULTS")
print("=" * 65)

for check_name, passed in all_checks.items():
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

M1B_PASSED = all(all_checks.values())

print("\nRebuilt beats:", len(rebuilt_check))
print("Records:", rebuilt_check["Record_ID"].nunique())
print("People:", rebuilt_check["Person_ID"].nunique())

print(
    "\nM1-B FINAL STATUS:",
    "PASS" if M1B_PASSED else "NOT PASSED"
)

In [ ]:

# ================================================================
# M1-B CELL 11 — SAVE VERIFIED BEAT TABLE AND AUDIT RESULTS
# ================================================================

print("=" * 70)
print("M1-B: SAVE FINAL OUTPUTS")
print("=" * 70)

OUTPUT_DIR = Path(
    "/kaggle/working/ECG_Project/01_M1_Verification"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save diagnostic information even if a check failed.
annotation_audit_df.to_csv(
    OUTPUT_DIR / "M1_annotation_audit.csv",
    index=False
)

record_audit_df.to_csv(
    OUTPUT_DIR / "M1_record_extraction_audit.csv",
    index=False
)

class_count_comparison.to_csv(
    OUTPUT_DIR / "M1_class_count_comparison.csv"
)

verification_results.to_csv(
    OUTPUT_DIR / "M1_verification_checks.csv",
    index=False
)

if not M1B_PASSED:

    if not mismatch_diagnostics.empty:
        mismatch_diagnostics.to_csv(
            OUTPUT_DIR / "M1_mismatch_diagnostics.csv",
            index=False
        )

    raise RuntimeError(
        "M1-B did not pass all validation checks. "
        "Diagnostic tables were saved. "
        "Do not use this beat table for M2."
    )

# Save the verified rebuilt beat table.
verified_path = (
    OUTPUT_DIR / "M1_rebuilt_verified_beat_table.csv.gz"
)

rebuilt_df.to_csv(
    verified_path,
    index=False,
    compression="gzip"
)

# Save overall class counts.
rebuilt_class_counts = (
    rebuilt_df["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
    .rename_axis("AAMI_Label")
    .reset_index(name="Beat_Count")
)

rebuilt_class_counts.to_csv(
    OUTPUT_DIR / "M1_verified_class_counts.csv",
    index=False
)

# Create a compact text report.
report_lines = [
    "M1-B ORIGINAL SIGNAL AND ANNOTATION RECONSTRUCTION",
    "=" * 65,
    "",
    f"Processed records: {len(PROCESSED_RECORD_IDS)}",
    f"Rebuilt beats: {len(rebuilt_df)}",
    f"Rebuilt people: {rebuilt_df['Person_ID'].nunique()}",
    "",
    "CLASS COUNTS:"
]

for class_name in AAMI_CLASSES:
    count = int(
        rebuilt_class_counts.loc[
            rebuilt_class_counts["AAMI_Label"] == class_name,
            "Beat_Count"
        ].iloc[0]
    )

    report_lines.append(f"{class_name}: {count}")

report_lines.extend([
    "",
    "VERIFICATION CHECKS:"
])

for _, row in verification_results.iterrows():
    report_lines.append(
        f"{row['Check']}: {'PASS' if row['PASS'] else 'FAIL'}"
    )

report_lines.extend([
    "",
    "FINAL GATE: PASS",
    "",
    "All 402 waveform values were verified against the existing",
    "processed dataset using record-specific waveform fingerprints.",
    "The comparison did not depend on row order.",
    "",
    f"Verified output: {verified_path}"
])

report_text = "\n".join(report_lines)

report_path = OUTPUT_DIR / "M1_FINAL_VERIFICATION_REPORT.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

manifest = {
    "phase": "M1",
    "status": "PASS",
    "records": int(rebuilt_df["Record_ID"].nunique()),
    "people": int(rebuilt_df["Person_ID"].nunique()),
    "beats": int(len(rebuilt_df)),
    "sampling_rate_hz": 360,
    "window_before": WINDOW_BEFORE,
    "window_after": WINDOW_AFTER,
    "window_length_per_lead": WINDOW_LENGTH,
    "waveform_values_per_beat": len(WAVEFORM_COLUMNS),
    "aami_class_counts": {
        str(row["AAMI_Label"]): int(row["Beat_Count"])
        for _, row in rebuilt_class_counts.iterrows()
    },
    "checks": {
        str(row["Check"]): bool(row["PASS"])
        for _, row in verification_results.iterrows()
    }
}

manifest_path = OUTPUT_DIR / "M1_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(report_text)

print("\nSAVED FILES:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(
        path.name,
        f"({path.stat().st_size / (1024**2):.2f} MB)"
    )

print("\nM1-B COMPLETE.")
print("No model was trained.")
print("No M0 prediction files or saved split assignments were changed.")


In [ ]:

# ================================================================
# M1-B FAILURE DIAGNOSTICS — ACCOUNT B
# ================================================================

print("=" * 70)
print("M1-B: IDENTIFY FAILED VALIDATION CHECKS")
print("=" * 70)

print("\n1. ALL VALIDATION CHECKS")
for check, result in all_checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print("\n2. CLASS COUNT COMPARISON")
display(class_count_comparison)

print("\n3. RECORD-LEVEL EXTRACTION SUMMARY")
display(
    record_audit_df[
        [
            "Record_ID",
            "Annotation_Beat_Candidates",
            "Accepted_Full_Windows",
            "Boundary_Windows_Skipped"
        ]
    ]
)

print("\n4. ANNOTATION PARSER ERRORS")
print("Total parser errors:", len(parser_errors_df))

if not parser_errors_df.empty:
    display(parser_errors_df.head(20))

print("\n5. WAVEFORM MISMATCH DIAGNOSTICS")

if not mismatch_diagnostics.empty:
    display(mismatch_diagnostics)
else:
    print("No record-level waveform mismatch table was generated.")

print("\n6. TOTALS")
print("Processed valid beats:", len(processed_valid))
print("Rebuilt beats:", len(rebuilt_df))
print("Records:", rebuilt_df["Record_ID"].nunique())
print("People:", rebuilt_df["Person_ID"].nunique())

print("\nDiagnostic inspection complete.")


In [ ]:
# M1-B diagnostic: inspect original labels and their AAMI mapping
# This cell only reads data; it does not modify the source dataset.

AAMI_MAP_CHECK = {
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V",
    "F": "F",
    "/": "Q", "f": "Q", "Q": "Q"
}

print("1. Original Label values in processed_df:")
display(processed_df["Label"].value_counts(dropna=False).to_frame("Count"))

print("\n2. AAMI counts after mapping the original labels:")
mapped_labels = (
    processed_df["Label"]
    .astype(str)
    .str.strip()
    .map(AAMI_MAP_CHECK)
)
display(mapped_labels.value_counts(dropna=False).to_frame("Count"))

print("\n3. Record 114: original processed rows:")
display(
    processed_df.loc[
        processed_df["Record_ID"].astype(int) == 114,
        ["Record_ID", "Label"]
    ].head(10)
)

print("\n4. Record 114: rebuilt rows:")
display(
    rebuilt_df.loc[
        rebuilt_df["Record_ID"].astype(int) == 114,
        ["Record_ID", "Sample_Index", "Original_Symbol", "AAMI_Label"]
    ].head(10)
)

In [ ]:
# ============================================================
# M1-B DIAGNOSTIC: RECORD 114 LEAD-BY-LEAD INSPECTION
# ============================================================

import os
import hashlib
import numpy as np
import pandas as pd
from collections import Counter

RECORD_TO_INSPECT = 114

# ------------------------------------------------------------
# 1. Locate the selected source signal file
# ------------------------------------------------------------

signal_path = next(
    (
        value for key, value in signal_files.items()
        if int(key) == RECORD_TO_INSPECT
    ),
    None
)

if signal_path is None:
    raise FileNotFoundError(
        "No selected signal file found for record 114."
    )

print("SELECTED SOURCE SIGNAL FILE")
print(signal_path)

# ------------------------------------------------------------
# 2. Read the original signal and inspect its columns
# ------------------------------------------------------------

source_signal = pd.read_csv(signal_path)

source_signal.columns = [
    str(c).strip().strip('"').strip().lower()
    for c in source_signal.columns
]

print("\nSOURCE SIGNAL SHAPE:", source_signal.shape)
print("\nSOURCE SIGNAL COLUMNS:")
print(source_signal.columns.tolist())

sample_col = next(
    (c for c in source_signal.columns
     if c.replace(" ", "") in ["sample#", "sample"]),
    None
)

mlii_col = next(
    (c for c in source_signal.columns if c.upper() == "MLII"),
    None
)

other_leads = [
    c for c in source_signal.columns
    if c not in [sample_col, mlii_col]
]

print("\nSample index column:", sample_col)
print("MLII column:", mlii_col)
print("Other signal columns:", other_leads)

if sample_col is None or mlii_col is None or len(other_leads) != 1:
    raise ValueError(
        "Record 114 does not have the expected sample, MLII "
        "and single second-lead column structure."
    )

print("\nFIRST FIVE SOURCE SIGNAL ROWS:")
display(
    source_signal[
        [sample_col, mlii_col] + other_leads
    ].head()
)

# ------------------------------------------------------------
# 3. Isolate record 114 in both datasets
# ------------------------------------------------------------

processed_114 = processed_valid[
    processed_valid["Record_ID"].astype(int)
    == RECORD_TO_INSPECT
].copy()

rebuilt_114 = rebuilt_check[
    rebuilt_check["Record_ID"].astype(int)
    == RECORD_TO_INSPECT
].copy()

print("\nPROCESSED BEATS:", len(processed_114))
print("REBUILT BEATS:", len(rebuilt_114))

# ------------------------------------------------------------
# 4. Compare each lead independently
# ------------------------------------------------------------

def lead_fingerprint_counter(dataframe, columns):
    counter = Counter()

    values = dataframe[columns].to_numpy(dtype=np.float64)

    for row in values:
        row = np.ascontiguousarray(row)
        digest = hashlib.blake2b(
            row.tobytes(),
            digest_size=16
        ).hexdigest()
        counter[digest] += 1

    return counter


lead_comparison = []

for lead_name, columns in [
    (
        "MLII",
        [f"MLII_{i}" for i in range(201)]
    ),
    (
        "Second lead",
        [f"Lead2_{i}" for i in range(201)]
    )
]:
    processed_counter = lead_fingerprint_counter(
        processed_114, columns
    )

    rebuilt_counter = lead_fingerprint_counter(
        rebuilt_114, columns
    )

    lead_comparison.append({
        "Lead": lead_name,
        "Processed_beats": sum(processed_counter.values()),
        "Rebuilt_beats": sum(rebuilt_counter.values()),
        "Missing_matches": sum(
            (processed_counter - rebuilt_counter).values()
        ),
        "Extra_matches": sum(
            (rebuilt_counter - processed_counter).values()
        ),
        "Exact_match": processed_counter == rebuilt_counter
    })

print("\nLEAD-BY-LEAD COMPARISON:")
display(pd.DataFrame(lead_comparison))

# ------------------------------------------------------------
# 5. Inspect the first reconstructed beat against source signal
# ------------------------------------------------------------

first_beat = rebuilt_114.iloc[0]
centre = int(first_beat["Sample_Index"])

start = centre - 100
end = centre + 101

print("\nFIRST REBUILT BEAT")
print("Sample index:", centre)
print("Original annotation:", first_beat["Original_Symbol"])
print("AAMI class:", first_beat["AAMI_Label"])
print("Window start:", start)
print("Window end (exclusive):", end)

if start < 0 or end > len(source_signal):
    print("WARNING: First beat window falls outside source signal.")
else:
    source_window = source_signal.iloc[start:end]

    print("\nSOURCE MLII: FIRST FIVE WINDOW VALUES")
    print(
        source_window[mlii_col]
        .to_numpy(dtype=float)[:5]
    )

    print("\nREBUILT MLII: FIRST FIVE WINDOW VALUES")
    print(
        first_beat[
            [f"MLII_{i}" for i in range(5)]
        ].to_numpy(dtype=float)
    )

    print("\nSOURCE SECOND LEAD:", other_leads[0])
    print(
        source_window[other_leads[0]]
        .to_numpy(dtype=float)[:5]
    )

    print("\nREBUILT Lead2: FIRST FIVE WINDOW VALUES")
    print(
        first_beat[
            [f"Lead2_{i}" for i in range(5)]
        ].to_numpy(dtype=float)
    )

print("\nRECORD 114 DIAGNOSTIC COMPLETE.")

In [ ]:
# ============================================================
# M1-B: RECORD 114 — ROBUST COLUMN DETECTION
# ============================================================

import re
import unicodedata
import numpy as np
import pandas as pd
import hashlib
from collections import Counter

RECORD_TO_INSPECT = 114

# ------------------------------------------------------------
# 1. Locate and load the source file
# ------------------------------------------------------------

signal_path = next(
    (
        value for key, value in signal_files.items()
        if int(key) == RECORD_TO_INSPECT
    ),
    None
)

if signal_path is None:
    raise FileNotFoundError("Source file for record 114 not found.")

source_signal = pd.read_csv(signal_path)

print("Source file:", signal_path)
print("Source shape:", source_signal.shape)

print("\nOriginal column names (including hidden characters):")
for c in source_signal.columns:
    print(repr(c))

# ------------------------------------------------------------
# 2. Identify columns using robust name normalisation
# ------------------------------------------------------------

def normalise_column_name(name):
    name = unicodedata.normalize("NFKC", str(name))
    name = name.casefold()
    return re.sub(r"[^a-z0-9#]", "", name)

column_lookup = {
    normalise_column_name(c): c
    for c in source_signal.columns
}

print("\nNormalised column names:")
for key, original in column_lookup.items():
    print(repr(original), "->", repr(key))

sample_col = (
    column_lookup.get("sample#")
    or column_lookup.get("sample")
)

mlii_col = column_lookup.get("mlii")

if sample_col is None:
    raise ValueError("Could not identify sample index column.")

if mlii_col is None:
    raise ValueError("Could not identify MLII column.")

second_leads = [
    c for c in source_signal.columns
    if c not in [sample_col, mlii_col]
]

if len(second_leads) != 1:
    raise ValueError(
        f"Expected one second lead, found: {second_leads}"
    )

second_lead_col = second_leads[0]

print("\nIDENTIFIED COLUMNS")
print("Sample index:", repr(sample_col))
print("MLII:", repr(mlii_col))
print("Second lead:", repr(second_lead_col))

display(
    source_signal[
        [sample_col, mlii_col, second_lead_col]
    ].head()
)

# ------------------------------------------------------------
# 3. Isolate record 114 from both datasets
# ------------------------------------------------------------

processed_114 = processed_valid[
    processed_valid["Record_ID"].astype(int) == 114
].copy()

rebuilt_114 = rebuilt_check[
    rebuilt_check["Record_ID"].astype(int) == 114
].copy()

print("\nProcessed beats:", len(processed_114))
print("Rebuilt beats:", len(rebuilt_114))

# ------------------------------------------------------------
# 4. Compare each lead independently
# ------------------------------------------------------------

def fingerprint_counter(df, columns):
    result = Counter()

    for values in df[columns].to_numpy(dtype=np.float64):
        values = np.ascontiguousarray(values)
        digest = hashlib.blake2b(
            values.tobytes(),
            digest_size=16
        ).hexdigest()
        result[digest] += 1

    return result


lead_results = []

for lead_name, columns in [
    ("MLII", [f"MLII_{i}" for i in range(201)]),
    ("Second lead", [f"Lead2_{i}" for i in range(201)])
]:
    p = fingerprint_counter(processed_114, columns)
    r = fingerprint_counter(rebuilt_114, columns)

    lead_results.append({
        "Lead": lead_name,
        "Missing_matches": sum((p - r).values()),
        "Extra_matches": sum((r - p).values()),
        "Exact_match": p == r
    })

print("\nLEAD-BY-LEAD COMPARISON")
display(pd.DataFrame(lead_results))

# ------------------------------------------------------------
# 5. Compare the first rebuilt beat directly with source signal
# ------------------------------------------------------------

first_beat = rebuilt_114.iloc[0]
centre = int(first_beat["Sample_Index"])

start = centre - 100
end = centre + 101

print("\nFIRST REBUILT BEAT")
print("Sample index:", centre)
print("Original symbol:", first_beat["Original_Symbol"])
print("AAMI label:", first_beat["AAMI_Label"])

if start < 0 or end > len(source_signal):
    print("First beat window is outside the source signal.")
else:
    source_window = source_signal.iloc[start:end]

    source_mlii = source_window[mlii_col].to_numpy(dtype=float)
    rebuilt_mlii = first_beat[
        [f"MLII_{i}" for i in range(201)]
    ].to_numpy(dtype=float)

    source_second = source_window[
        second_lead_col
    ].to_numpy(dtype=float)

    rebuilt_second = first_beat[
        [f"Lead2_{i}" for i in range(201)]
    ].to_numpy(dtype=float)

    print("\nMLII source vs rebuilt:")
    print("Exact match:", np.array_equal(source_mlii, rebuilt_mlii))
    print("Source first 5:", source_mlii[:5])
    print("Rebuilt first 5:", rebuilt_mlii[:5])

    print("\nSecond lead source vs rebuilt:")
    print("Exact match:", np.array_equal(source_second, rebuilt_second))
    print("Source first 5:", source_second[:5])
    print("Rebuilt first 5:", rebuilt_second[:5])

print("\nRECORD 114 DIAGNOSTIC COMPLETE.")

In [ ]:
# ============================================================
# M1-B: DIRECT COMPARISON OF RECORD 114 BEATS
# ============================================================

import numpy as np
import pandas as pd

record_id = 114

mlii_cols = [f"MLII_{i}" for i in range(201)]
lead2_cols = [f"Lead2_{i}" for i in range(201)]
all_wave_cols = mlii_cols + lead2_cols

# Isolate the two versions of record 114
p114 = processed_valid[
    processed_valid["Record_ID"].astype(int) == record_id
].copy().reset_index(drop=True)

r114 = rebuilt_check[
    rebuilt_check["Record_ID"].astype(int) == record_id
].copy().reset_index(drop=True)

print("Processed beats:", len(p114))
print("Rebuilt beats:", len(r114))

# ------------------------------------------------------------
# 1. Compare the first reconstructed beat against all
#    processed beats from the same record.
# ------------------------------------------------------------

rebuilt_first = r114.iloc[0]
rebuilt_wave = rebuilt_first[all_wave_cols].to_numpy(dtype=float)

processed_waves = p114[all_wave_cols].to_numpy(dtype=float)

exact_matches = np.where(
    np.all(processed_waves == rebuilt_wave, axis=1)
)[0]

print("\nFIRST REBUILT BEAT")
print("Sample index:", rebuilt_first["Sample_Index"])
print("Original symbol:", rebuilt_first["Original_Symbol"])
print("Exact matching processed rows:", len(exact_matches))

if len(exact_matches):
    print("Matching processed row positions:", exact_matches.tolist())

# ------------------------------------------------------------
# 2. Find the closest processed beat to the first rebuilt beat.
# ------------------------------------------------------------

absolute_differences = np.abs(
    processed_waves - rebuilt_wave
)

mean_absolute_difference = absolute_differences.mean(axis=1)
maximum_absolute_difference = absolute_differences.max(axis=1)

closest_position = int(np.argmin(mean_absolute_difference))

print("\nCLOSEST PROCESSED BEAT")
print("Position within processed record:", closest_position)
print("Original dataframe index:", p114.index[closest_position])
print(
    "Mean absolute difference:",
    mean_absolute_difference[closest_position]
)
print(
    "Maximum absolute difference:",
    maximum_absolute_difference[closest_position]
)

# ------------------------------------------------------------
# 3. Compare each lead separately for that closest row.
# ------------------------------------------------------------

processed_first = p114.iloc[closest_position]

for lead_name, columns in [
    ("MLII", mlii_cols),
    ("Lead2", lead2_cols)
]:
    source_values = rebuilt_first[columns].to_numpy(dtype=float)
    processed_values = processed_first[columns].to_numpy(dtype=float)

    print(f"\n{lead_name} — first 10 values")
    print("Rebuilt:  ", source_values[:10])
    print("Processed:", processed_values[:10])

    print("Exact match:", np.array_equal(
        source_values, processed_values
    ))

    print(
        "Mean absolute difference:",
        np.mean(np.abs(source_values - processed_values))
    )

    print(
        "Maximum absolute difference:",
        np.max(np.abs(source_values - processed_values))
    )

# ------------------------------------------------------------
# 4. Show the first few rows of each version for context.
# ------------------------------------------------------------

print("\nFIRST FIVE REBUILT BEATS")
display(
    r114[
        ["Sample_Index", "Original_Symbol", "AAMI_Label"]
    ].head()
)

print("\nFIRST FIVE PROCESSED BEATS")
display(
    p114[
        ["Record_ID", "Label", "AAMI_Label"]
    ].head()
)

print("\nDIRECT COMPARISON COMPLETE.")

In [ ]:
# ============================================================
# M1-B: CHECK ALL SOURCE COPIES OF RECORD 114
# ============================================================

import os
import re
import hashlib
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# ------------------------------------------------------------
# 1. Find every file named 114.csv in Kaggle inputs
# ------------------------------------------------------------

candidates = []

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if filename.lower() == "114.csv":
            candidates.append(os.path.join(root, filename))

print("Number of 114.csv files found:", len(candidates))

for path in candidates:
    print(path)

if not candidates:
    raise FileNotFoundError("No 114.csv file found under /kaggle/input.")

# ------------------------------------------------------------
# 2. Get the first processed waveform for record 114
# ------------------------------------------------------------

p114 = processed_valid[
    processed_valid["Record_ID"].astype(int) == 114
].reset_index(drop=True)

target_mlii = p114.iloc[0][
    [f"MLII_{i}" for i in range(201)]
].to_numpy(dtype=float)

target_lead2 = p114.iloc[0][
    [f"Lead2_{i}" for i in range(201)]
].to_numpy(dtype=float)

print("\nFirst processed beat label:", p114.iloc[0]["AAMI_Label"])
print("Processed first MLII values:", target_mlii[:10])
print("Processed first Lead2 values:", target_lead2[:10])

# ------------------------------------------------------------
# 3. Search each candidate for an exact matching 201-sample
#    window on MLII and its second lead.
# ------------------------------------------------------------

def normalise_name(name):
    return re.sub(
        r"[^a-z0-9#]",
        "",
        str(name).casefold()
    )

def find_exact_window(signal, target, prefix_length=12):

    signal = np.asarray(signal, dtype=float)
    target = np.asarray(target, dtype=float)

    if len(signal) < len(target):
        return []

    # First find possible positions using the first 12 values.
    possible = np.flatnonzero(
        np.all(
            sliding_window_view(signal, prefix_length)
            == target[:prefix_length],
            axis=1
        )
    )

    # Verify all 201 values at each possible position.
    exact = []

    for start in possible:
        if np.array_equal(
            signal[start:start + len(target)],
            target
        ):
            exact.append(int(start))

    return exact


candidate_results = []

for path in candidates:

    print("\nInspecting:", path)

    try:
        df = pd.read_csv(path)

        lookup = {
            normalise_name(c): c
            for c in df.columns
        }

        sample_col = (
            lookup.get("sample#")
            or lookup.get("sample")
        )

        mlii_col = lookup.get("mlii")

        if sample_col is None or mlii_col is None:
            candidate_results.append({
                "File": path,
                "Status": "Required columns not found",
                "MLII_exact_start": None,
                "Second_lead": None,
                "Second_lead_exact_start": None,
                "Same_start_both_leads": False
            })
            continue

        other_leads = [
            c for c in df.columns
            if c not in [sample_col, mlii_col]
        ]

        mlii_values = pd.to_numeric(
            df[mlii_col], errors="raise"
        ).to_numpy(dtype=float)

        mlii_starts = find_exact_window(
            mlii_values, target_mlii
        )

        if len(other_leads) == 1:
            second_col = other_leads[0]

            second_values = pd.to_numeric(
                df[second_col], errors="raise"
            ).to_numpy(dtype=float)

            second_starts = find_exact_window(
                second_values, target_lead2
            )

            same_start = sorted(
                set(mlii_starts) & set(second_starts)
            )

            candidate_results.append({
                "File": path,
                "Status": "Inspected",
                "MLII_exact_start": mlii_starts,
                "Second_lead": second_col,
                "Second_lead_exact_start": second_starts,
                "Same_start_both_leads": bool(same_start),
                "Matching_window_start": same_start
            })

        else:
            candidate_results.append({
                "File": path,
                "Status": f"Unexpected number of other leads: {len(other_leads)}",
                "MLII_exact_start": mlii_starts,
                "Second_lead": str(other_leads),
                "Second_lead_exact_start": None,
                "Same_start_both_leads": False
            })

    except Exception as exc:
        candidate_results.append({
            "File": path,
            "Status": f"ERROR: {exc}",
            "MLII_exact_start": None,
            "Second_lead": None,
            "Second_lead_exact_start": None,
            "Same_start_both_leads": False
        })

# ------------------------------------------------------------
# 4. Display findings
# ------------------------------------------------------------

results_df = pd.DataFrame(candidate_results)

print("\n" + "=" * 65)
print("RECORD 114 SOURCE-COPY COMPARISON")
print("=" * 65)

display(results_df)

print("\nDiagnostic complete. No data was modified.")

In [ ]:
# M1-B: Verify whether record 114 has reversed lead blocks

import numpy as np
import pandas as pd

mlii_cols = [f"MLII_{i}" for i in range(201)]
lead2_cols = [f"Lead2_{i}" for i in range(201)]

p114 = processed_valid[
    processed_valid["Record_ID"].astype(int) == 114
].reset_index(drop=True)

r114 = rebuilt_check[
    rebuilt_check["Record_ID"].astype(int) == 114
].reset_index(drop=True)

# Existing processed data, original lead order
processed_original = p114[
    mlii_cols + lead2_cols
].to_numpy(dtype=float)

# Existing processed data, with the two lead blocks swapped
processed_swapped = p114[
    lead2_cols + mlii_cols
].to_numpy(dtype=float)

# Correctly identified leads from original signal
rebuilt = r114[
    mlii_cols + lead2_cols
].to_numpy(dtype=float)

print("Processed beats:", len(p114))
print("Rebuilt beats:", len(r114))

print(
    "\nExact row-by-row match before swapping:",
    np.array_equal(processed_original, rebuilt)
)

print(
    "Exact row-by-row match after swapping:",
    np.array_equal(processed_swapped, rebuilt)
)

print(
    "\nNumber of matching beats after swapping:",
    np.sum(np.all(processed_swapped == rebuilt, axis=1)),
    "out of", len(rebuilt)
)

print(
    "\nAAMI labels match row by row:",
    np.array_equal(
        p114["AAMI_Label"].to_numpy(),
        r114["AAMI_Label"].to_numpy()
    )
)